In [22]:
'''
Overview
this notebook is a working example of scraping ufc fight stats one at a time
the code is broken down into small steps that can be run in parts to view and verify results at each step
this is useful for testing and debugging the code as the ufc stats website may be update, breaking the code


scrape ufc fight stats
get all event details, name, url, date, location for all ufc events
for each event, get fight details all fights on card
parse each fight to get fight stats of both fighters

additional scraping of fighter's details, name, nickname, url
for each fighter, get their tale of the tape
'''

"\nOverview\nthis notebook is a working example of scraping ufc fight stats one at a time\nthe code is broken down into small steps that can be run in parts to view and verify results at each step\nthis is useful for testing and debugging the code as the ufc stats website may be update, breaking the code\n\n\nscrape ufc fight stats\nget all event details, name, url, date, location for all ufc events\nfor each event, get fight details all fights on card\nparse each fight to get fight stats of both fighters\n\nadditional scraping of fighter's details, name, nickname, url\nfor each fighter, get their tale of the tape\n"

In [23]:
# imports
import pandas as pd
import numpy as np
import os
import re
import requests
from bs4 import BeautifulSoup
import itertools

# import library
import scrape_ufc_stats_library as LIB
import importlib
importlib.reload(LIB)

# import configs
import yaml
config = yaml.safe_load(open('scrape_ufc_stats_config.yaml'))

# Parse Event Details
Includes:
<br>
Event
<br>
URL
<br>
Date
<br>
Location
<br>

In [24]:
# define url to parse
url = 'http://ufcstats.com/statistics/events/completed' # first page
# url = 'http://ufcstats.com/statistics/events/completed?page=all' # all pages

# get soup
soup = LIB.get_soup(url)

In [25]:
# parse event details
event_details_df = LIB.parse_event_details(soup)

# show event details
display(event_details_df)

,EVENT,URL,DATE,LOCATION
0,UFC Fight Night: Della Maddalena vs. Prates,http://ufcstats.com/event-details/872b018076f8...,"May 02, 2026","Perth, Western Australia, Australia"
1,UFC Fight Night: Sterling vs. Zalal,http://ufcstats.com/event-details/e60d773a0a42...,"April 25, 2026","Las Vegas, Nevada, USA"
2,UFC Fight Night: Burns vs. Malott,http://ufcstats.com/event-details/c3ac8d0da7b0...,"April 18, 2026","Winnipeg, Manitoba, Canada"
3,UFC 327: Prochazka vs. Ulberg,http://ufcstats.com/event-details/f3eb664db7fb...,"April 11, 2026","Miami, Florida, USA"
4,UFC Fight Night: Moicano vs. Duncan,http://ufcstats.com/event-details/9a70f67ad218...,"April 04, 2026","Las Vegas, Nevada, USA"
5,UFC Fight Night: Adesanya vs. Pyfer,http://ufcstats.com/event-details/5c38639f860a...,"March 28, 2026","Seattle, Washington, USA"
6,UFC Fight Night: Evloev vs. Murphy,http://ufcstats.com/event-details/69108cb8b32e...,"March 21, 2026","London, England, United Kingdom"
7,UFC Fight Night: Emmett vs. Vallejos,http://ufcstats.com/event-details/babc6b574533...,"March 14, 2026","Las Vegas, Nevada, USA"
8,UFC 326: Holloway vs. Oliveira 2,http://ufcstats.com/event-details/15ec018d1447...,"March 07, 2026","Las Vegas, Nevada, USA"
9,UFC Fight Night: Moreno vs. Kavanagh,http://ufcstats.com/event-details/0cfbbfa0ba6d...,"February 28, 2026","Mexico City, Distrito Federal, Mexico"


# Parse Fight Details
Includes:
<br>
Event
<br>
Bout
<br>
URL

In [26]:
# parse one event for fight details

# define url to parse
url = 'http://ufcstats.com/event-details/509697e08673d2e5'
# get soup
soup = LIB.get_soup(url)

In [27]:
# parse fight links
fight_details_df = LIB.parse_fight_details(soup, url)

# show fight links
display(fight_details_df)

,EVENT,BOUT,URL,EVENT_URL
0,UFC Fight Night: Font vs. Aldo,Rob Font vs. Jose Aldo,http://ufcstats.com/fight-details/3109d1151f14...,http://ufcstats.com/event-details/509697e08673...
1,UFC Fight Night: Font vs. Aldo,Brad Riddell vs. Rafael Fiziev,http://ufcstats.com/fight-details/a38648a1c190...,http://ufcstats.com/event-details/509697e08673...
2,UFC Fight Night: Font vs. Aldo,Jimmy Crute vs. Jamahal Hill,http://ufcstats.com/fight-details/b3847772199e...,http://ufcstats.com/event-details/509697e08673...
3,UFC Fight Night: Font vs. Aldo,Clay Guida vs. Leonardo Santos,http://ufcstats.com/fight-details/5d64044b9850...,http://ufcstats.com/event-details/509697e08673...
4,UFC Fight Night: Font vs. Aldo,Brendan Allen vs. Chris Curtis,http://ufcstats.com/fight-details/9e5bbbacf1d9...,http://ufcstats.com/event-details/509697e08673...
5,UFC Fight Night: Font vs. Aldo,Alex Morono vs. Mickey Gall,http://ufcstats.com/fight-details/657b7da9c893...,http://ufcstats.com/event-details/509697e08673...
6,UFC Fight Night: Font vs. Aldo,Maki Pitolo vs. Dusko Todorovic,http://ufcstats.com/fight-details/723a8a8c82ca...,http://ufcstats.com/event-details/509697e08673...
7,UFC Fight Night: Font vs. Aldo,Manel Kape vs. Zhalgas Zhumagulov,http://ufcstats.com/fight-details/376dddfa82de...,http://ufcstats.com/event-details/509697e08673...
8,UFC Fight Night: Font vs. Aldo,Bryan Barberena vs. Darian Weeks,http://ufcstats.com/fight-details/405bded9cc7a...,http://ufcstats.com/event-details/509697e08673...
9,UFC Fight Night: Font vs. Aldo,Cheyanne Vlismas vs. Mallory Martin,http://ufcstats.com/fight-details/3fc8afbb23b9...,http://ufcstats.com/event-details/509697e08673...


# Parse Fight Results and Stats

In [28]:
# define url to parse
# various types of fights
# url = 'http://ufcstats.com/fight-details/4b7ec02b39fc6f70' # one round finish
# url = 'http://ufcstats.com/fight-details/8b3b38167060b1d7' # three rounds decision
# url = 'http://ufcstats.com/fight-details/b22eab3aa1522f40' # three rounds finish
# url = 'http://ufcstats.com/fight-details/3109d1151f149aaf' # five rounds decision
# url = 'http://ufcstats.com/fight-details/d93c8c77e1091a16' # no stats
# url = 'http://ufcstats.com/fight-details/c63edd25d2201a46' # draw
url = 'http://ufcstats.com/fight-details/37cb7ce0f0b70640' # point deduction

# get soup
soup = LIB.get_soup(url)

### Parse Fight Results

Includes:
<br>
Event
<br>
Bout
<br>
Weightclass
<br>
Method
<br>
Round
<br>
Time
<br>
Time Format
<br>
Referee
<br>
Details
<br>

In [ ]:
# parse fight results from soup
fight_results = LIB.parse_fight_results(soup)
# append fight url
fight_results.append('URL:'+url)
fight_results.append('EVENT_URL:') # leaving blank for example

# show fight results
fight_results

['UFC 33: Victory in Vegas ',
 'Dave Menne',
 'Gil Castillo',
 'W',
 'L',
 'UFC Middleweight Title Bout',
 'Method: Decision - Unanimous ',
 'Round:5',
 'Time:5:00',
 'Time format:5 Rnd (5-5-5-5-5)',
 'Referee:John McCarthy',
 'Details:Point Deducted: Illegal Knee by MenneTony Weeks 45 - 49.Doug Crosby 42 - 49.Jeff Mullen 44 - 49.',
 'URL:http://ufcstats.com/fight-details/37cb7ce0f0b70640',
 'EVENT_URL:']

In [31]:
# organise fight results
fight_results_df = LIB.organise_fight_results(fight_results, config['fight_results_column_names'])

# show fight results
display(fight_results_df)

,EVENT,BOUT,OUTCOME,WEIGHTCLASS,METHOD,ROUND,TIME,TIME FORMAT,REFEREE,DETAILS,URL,EVENT_URL
0,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,W/L,UFC Middleweight Title Bout,Decision - Unanimous,5,5:00,5 Rnd (5-5-5-5-5),John McCarthy,Point Deducted: Illegal Knee by MenneTony Week...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,


### Parse Fight Stats
Includes:
Event
<br>
Bout
<br>
Round
<br>
Fighter
<br>
Kd
<br>
Sig.Str.
<br>
Sig.Str. %
<br>
Total Str.
<br>
Td
<br>
Td %
<br>
Sub.Att
<br>
Rev.
<br>
Ctrl
<br>
Head
<br>
Body
<br>
Leg
<br>
Distance
<br>
Clinch
<br>
Ground
<br>

In [32]:
# parse full fight stats for both fighters
fighter_a_stats, fighter_b_stats = LIB.parse_fight_stats(soup)

# show fighter stats
print(fighter_a_stats[:20])

['Dave Menne', '0', '70 of 125', '56%', '180 of 241', '1 of 1', '100%', '4', '2', '7:54', 'Dave Menne', '0', '15 of 22', '68%', '45 of 52', '1 of 1', '100%', '1', '0', '2:02']


In [33]:
# organise stats extracted from soup
fighter_a_stats_clean = LIB.organise_fight_stats(fighter_a_stats)
fighter_b_stats_clean = LIB.organise_fight_stats(fighter_b_stats)

# show organised stats
print(fighter_a_stats_clean[:2])

[['Dave Menne', '0', '70 of 125', '56%', '180 of 241', '1 of 1', '100%', '4', '2', '7:54'], ['Dave Menne', '0', '15 of 22', '68%', '45 of 52', '1 of 1', '100%', '1', '0', '2:02']]


In [34]:
# convert list of fighter stats into a structured dataframe
fighter_a_stats_df = LIB.convert_fight_stats_to_df(fighter_a_stats_clean, config['totals_column_names'], config['significant_strikes_column_names'])
fighter_b_stats_df = LIB.convert_fight_stats_to_df(fighter_b_stats_clean, config['totals_column_names'], config['significant_strikes_column_names'])

# show stats df
display(fighter_a_stats_df)

,ROUND,FIGHTER,KD,SIG.STR.,SIG.STR. %,TOTAL STR.,TD,TD %,SUB.ATT,REV.,CTRL,HEAD,BODY,LEG,DISTANCE,CLINCH,GROUND
0,Round 1,Dave Menne,0,15 of 22,68%,45 of 52,1 of 1,100%,1,0,2:02,5 of 9,8 of 10,2 of 3,2 of 4,8 of 10,5 of 8
1,Round 2,Dave Menne,0,6 of 15,40%,34 of 45,0 of 0,---,2,1,3:06,3 of 12,2 of 2,1 of 1,2 of 5,1 of 2,3 of 8
2,Round 3,Dave Menne,0,14 of 29,48%,29 of 47,0 of 0,---,0,1,0:14,7 of 21,3 of 3,4 of 5,10 of 24,4 of 5,0 of 0
3,Round 4,Dave Menne,0,21 of 41,51%,24 of 44,0 of 0,---,1,0,0:07,13 of 31,3 of 3,5 of 7,7 of 21,13 of 19,1 of 1
4,Round 5,Dave Menne,0,14 of 18,77%,48 of 53,0 of 0,---,0,0,2:25,10 of 13,0 of 0,4 of 5,3 of 5,4 of 6,7 of 7


In [35]:
# combine fighter stats into one (note: not including event URL here)
fight_stats = LIB.combine_fighter_stats_dfs(fighter_a_stats_df, fighter_b_stats_df, soup, url, '')

# show fight stats
display(fight_stats)

/Users/nolanmoak/Projects/scrapyard/scrapyard-ufc-import/Stats/scrape_ufc_stats_library.py:425: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  
/Users/nolanmoak/Projects/scrapyard/scrapyard-ufc-import/Stats/scrape_ufc_stats_library.py:427: Fu

,EVENT,BOUT,ROUND,FIGHTER,KD,SIG.STR.,SIG.STR. %,TOTAL STR.,TD,TD %,...,CTRL,HEAD,BODY,LEG,DISTANCE,CLINCH,GROUND,FIGHTER_URL,FIGHT_URL,EVENT_URL
0,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 1,Dave Menne,0,15 of 22,68%,45 of 52,1 of 1,100%,...,2:02,5 of 9,8 of 10,2 of 3,2 of 4,8 of 10,5 of 8,http://ufcstats.com/fighter-details/275cafc9de...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
1,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 2,Dave Menne,0,6 of 15,40%,34 of 45,0 of 0,---,...,3:06,3 of 12,2 of 2,1 of 1,2 of 5,1 of 2,3 of 8,http://ufcstats.com/fighter-details/275cafc9de...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
2,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 3,Dave Menne,0,14 of 29,48%,29 of 47,0 of 0,---,...,0:14,7 of 21,3 of 3,4 of 5,10 of 24,4 of 5,0 of 0,http://ufcstats.com/fighter-details/275cafc9de...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
3,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 4,Dave Menne,0,21 of 41,51%,24 of 44,0 of 0,---,...,0:07,13 of 31,3 of 3,5 of 7,7 of 21,13 of 19,1 of 1,http://ufcstats.com/fighter-details/275cafc9de...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
4,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 5,Dave Menne,0,14 of 18,77%,48 of 53,0 of 0,---,...,2:25,10 of 13,0 of 0,4 of 5,3 of 5,4 of 6,7 of 7,http://ufcstats.com/fighter-details/275cafc9de...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
0,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 1,Gil Castillo,0,0 of 0,---,22 of 22,1 of 4,25%,...,2:27,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0,http://ufcstats.com/fighter-details/31ceaf0e67...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
1,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 2,Gil Castillo,0,0 of 0,---,15 of 15,3 of 5,60%,...,1:04,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0,http://ufcstats.com/fighter-details/31ceaf0e67...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
2,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 3,Gil Castillo,0,4 of 11,36%,38 of 45,1 of 3,33%,...,2:48,2 of 9,2 of 2,0 of 0,4 of 11,0 of 0,0 of 0,http://ufcstats.com/fighter-details/31ceaf0e67...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
3,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 4,Gil Castillo,0,4 of 12,33%,21 of 29,1 of 2,50%,...,3:12,3 of 10,1 of 2,0 of 0,3 of 11,1 of 1,0 of 0,http://ufcstats.com/fighter-details/31ceaf0e67...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,
4,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 5,Gil Castillo,0,1 of 2,50%,14 of 15,1 of 5,20%,...,2:07,0 of 1,0 of 0,1 of 1,0 of 1,1 of 1,0 of 0,http://ufcstats.com/fighter-details/31ceaf0e67...,http://ufcstats.com/fight-details/37cb7ce0f0b7...,


# Parse Fighter Details

In [36]:
# define url to parse
url = 'http://ufcstats.com/statistics/fighters?char=b&page=all' # 'a' last names
url = 'http://ufcstats.com/statistics/fighters?char=b&page=all' # 'b' last names

In [37]:
# get soup
soup = LIB.get_soup(url)

In [38]:
# parse fighter details
fighter_details_df = LIB.parse_fighter_details(soup, config['fighter_details_column_names'])

# show fighter details
display(fighter_details_df)

,FIRST,LAST,NICKNAME,URL
0,Niklas,Backstrom,,http://ufcstats.com/fighter-details/39cc64bf0a...
1,Seth,Baczynski,The Polish Pistola,http://ufcstats.com/fighter-details/fd5b6598a3...
2,Abdul Azeem,Badakhshi,The Afghan Lion,http://ufcstats.com/fighter-details/968764372c...
3,Ryan,Bader,Darth,http://ufcstats.com/fighter-details/c0ab242c40...
4,Izabela,Badurek,,http://ufcstats.com/fighter-details/4fcf6e0c4e...
...,...,...,...,...
316,JP,Buys,Young Savage,http://ufcstats.com/fighter-details/0b36224780...
317,Dennis,Buzukja,The Great,http://ufcstats.com/fighter-details/c4d039123e...
318,Charles,Byrd,Kid Dynamite,http://ufcstats.com/fighter-details/76ee3d666c...
319,Steve,Byrnes,The Sergeant,http://ufcstats.com/fighter-details/4361245697...


# Parse Fighter Tale Of The Tape

In [39]:
# define url to parse
url = 'http://ufcstats.com/fighter-details/93fe7332d16c6ad9' # '--' and '' present
url = 'http://ufcstats.com/fighter-details/b361180739bed4b0' # '--' present
url = 'http://ufcstats.com/fighter-details/1897b7b913736a7c' # no first name
url = 'http://ufcstats.com/fighter-details/d0f3959b4a9747e6' # all tott present

In [40]:
# get soup
soup = LIB.get_soup(url)

In [41]:
# parse fighter tale of the tape
fighter_tott = LIB.parse_fighter_tott(soup)

# show tale of the tape
fighter_tott

['Fighter:Jose Aldo',
 'Height:5\' 7"',
 'Weight:135 lbs.',
 'Reach:70"',
 'STANCE:Orthodox',
 'DOB:Sep 09, 1986']

In [42]:
# create empty df to store fighters' tale of the tape
all_fighter_tott_df = pd.DataFrame(columns=config['fighter_tott_column_names'])

# organise fighter tale of the tape
fighter_tott_df = LIB.organise_fighter_tott(fighter_tott, config['fighter_tott_column_names'], url)

# show fighter tale of the tape
display(fighter_tott_df)

,FIGHTER,HEIGHT,WEIGHT,REACH,STANCE,DOB,URL
0,Jose Aldo,"5' 7""",135 lbs.,"70""",Orthodox,"Sep 09, 1986",http://ufcstats.com/fighter-details/d0f3959b4a...
